# Telco churn: EDA and feature analysis

Objective: inspect the actual local CSV and assess data quality and candidate predictors without using held-out test relationships. This is exploration, not a production service. The source is the IBM Telco Customer Churn dataset linked in the supplied plan. Local provenance/license must be confirmed before redistribution. No raw records or identifiers are displayed.

Run from the project root or notebooks directory after installing `.[dev,training,analysis]`. Set `CHURN_DATA_PATH` to override the default project-relative CSV path.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from churn.data import dataset_path, load_dataset, partitions
from churn.exploration import create_eda

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from project root or notebooks/"
frame, quality = load_dataset(dataset_path(ROOT))
splits = partitions(frame)
train_x, train_y = splits["train"]
print("Verified shape:", quality["shape"])
print("Target and class counts:", quality["target"], quality["target_counts"])
display(pd.DataFrame(quality["columns"]).T)
EDA_REPORT_DIR = ROOT / os.getenv("CHURN_EDA_REPORT_DIR", "reports/milestone3/eda")

## Leakage boundary and quality decisions

Seed 42; stratified 70/15/15 train/validation/test split. Only schema, data quality, and overall target counts use the full file. All exploratory target relationships and numeric distributions below use training only. Test outcomes are not explored for selection.

Exclude customerID as a unique identifier and separate Churn before preprocessing. Parse TotalCharges blanks as missing; retain rows and fit median imputation inside training folds. SeniorCitizen is categorical. No outlier deletion or target encoding. Snapshot timing is unverified: ensure predictors exist before the churn outcome before operational use.

In [ ]:
eda = create_eda(train_x, train_y, quality, EDA_REPORT_DIR)
display(Markdown((EDA_REPORT_DIR / "eda_summary.md").read_text(encoding="utf-8")))

## Numerical and categorical relationships

Histograms are normalized within class; boxplots hide individual outliers. Churn-rate bars show fractions rather than majority-class counts. Pearson measures linear numeric association and Spearman monotonic association. Neither establishes causation or suffices for selecting features.

In [ ]:
for name in [
    "target_distribution",
    "numeric_distributions",
    "categorical_churn_rates",
    "correlations",
]:
    display(Image(filename=str(EDA_REPORT_DIR / f"figures/{name}.png")))

In [ ]:
from churn.data import NUMERIC, numeric_frame
from churn.exploration import categorical_rates

print("Training numeric means by target (0=No, 1=Yes):")
display(numeric_frame(train_x[NUMERIC]).groupby(train_y).mean())
for feature in ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport"]:
    display(categorical_rates(train_x, train_y, feature).assign(feature=feature))

## Feature analysis and interpretation

Read the training-only means and category rates above: differences indicate potential ranking signal, not causal effects. High-rate categories suggest retention hypotheses only after considering sample size and confounding. Tenure and accumulated charges encode related exposure; service categories also overlap.

Notebook 02 compares four models with stratified CV and a supervised L1 selector fitted inside each fold. Validation permutation importance ranks original columns; foldwise importance and top-five frequency assess stability. Retain all encoded features unless the selector's mean paired CV AP improvement exceeds one paired standard error. This is a heuristic, not a significance test. Correlated predictors can share importance; no feature is removed solely for correlation.

gender and SeniorCitizen raise potential fairness risks and remain provisional exploratory inputs. Validate temporal availability and subgroups before production. Aggregate tables and figures are saved in reports/eda_summary.md; no dataset records are exported.

Milestone 3 reruns save EDA under reports/milestone3/eda/; historical baseline reports are preserved. Notebook 02 now validates the previously CV-tuned configurations and performs validation-only calibration/threshold analysis with MLflow.